# 01 - Fuentes de datos y construcción del dataset documental

## Objetivo del notebook

Este notebook tiene como objetivo construir una base documental fiable para el proyecto de Machine Learning sobre los documentos desclasificados del 23-F.

La pregunta principal que debe responder este notebook es:

> ¿Qué se puede analizar realmente con estos documentos?

Antes de aplicar modelos, necesitamos construir y validar el corpus documental. Para ello, este notebook no empieza directamente con modelos, sino con la creación de una base de datos trazable que permita saber:

- qué documentos existen;
- qué metadatos tiene cada documento;
- qué textos completos pueden utilizarse;
- qué PDFs pueden localizarse o descargarse;
- qué diferencias existen entre las fuentes disponibles;
- qué limitaciones presenta el corpus;
- qué mini casos de uso son viables para la entrega final.

## Productos de datos que se quieren construir

El objetivo técnico del notebook es generar varios productos intermedios. No todos se construyen al principio: algunos son resultado de comparar y validar fuentes.

### 1. Inventario documental de RTVE

Tabla con los metadatos principales de los documentos localizados en el Buscador RTVE 23-F.

Columnas previstas:

- `doc_id`: identificador único del documento.
- `source`: fuente principal del registro.
- `title`: título del documento.
- `pages`: número de páginas.
- `summary`: resumen disponible en RTVE.
- `keywords`: palabras clave disponibles en RTVE.
- `detail_url`: enlace a la página de detalle del documento.
- `pdf_url`: enlace al PDF o documento original, si está disponible.

Este inventario no incluirá el texto completo para evitar crear una tabla demasiado pesada.

### 2. Tabla de textos completos de RTVE

Tabla separada con el texto completo asociado a cada documento.

Columnas previstas:

- `doc_id`: identificador único del documento.
- `text_full`: texto completo del documento.
- `text_length_chars`: longitud del texto en caracteres.
- `text_length_words`: longitud aproximada del texto en palabras.
- `extraction_source`: origen del texto utilizado, por ejemplo, texto HTML de RTVE, OCR o extracción desde PDF.

Además, cuando sea útil, se guardará una copia individual de cada texto en formato `.txt` dentro de `data/raw/texts_rtve/`.

### 3. PDFs de RTVE

Carpeta con los PDFs u originales digitalizados asociados a los documentos de RTVE.

Los archivos se guardarán en:

`data/raw/pdfs_rtve/`

Además, se creará un manifiesto con la relación entre `doc_id`, URL original y ruta local del archivo.

### 4. Inventario oficial de La Moncloa

La Moncloa se utilizará como fuente institucional de contraste. El objetivo no es descargar todos sus PDFs ni duplicar archivos, sino construir un inventario oficial que permita verificar la cobertura de RTVE.

Se creará, si procede:

- `data/interim/inventory_moncloa.csv`

Este inventario se utilizará para comprobar si existen documentos en La Moncloa que no aparezcan en RTVE.

### 5. Comparación RTVE vs La Moncloa

Una vez construidos los inventarios, se compararán ambas fuentes para comprobar:

- qué documentos aparecen en RTVE y en La Moncloa;
- qué documentos aparecen solo en RTVE;
- qué documentos aparecen solo en La Moncloa;
- si hay documentos de La Moncloa que deban incorporarse al corpus final;
- si las diferencias afectan a la cobertura del análisis.

Solo en el caso de detectar documentos presentes en La Moncloa y ausentes en RTVE se valorará descargar esos PDFs concretos y extraer su texto para incorporarlos al corpus.

### 6. Tabla final de documentos validados

Después de comparar y validar las fuentes, se construirá una tabla final:

`data/interim/documents_master.csv`

Esta tabla no será el punto de partida del análisis, sino el resultado de la comparación entre fuentes. Su función será definir qué documentos forman parte del corpus final de trabajo antes de pasar a limpieza, EDA y modelado.

## Fuentes que se evaluarán

En esta fase se analizarán tres fuentes:

1. **[Buscador RTVE 23-F](https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/)**  
   Fuente principal indicada en el enunciado. Se usará principalmente para obtener metadatos documentales y texto completo: título, páginas, resumen, palabras clave, página de detalle y texto transcrito.

2. **[Lista íntegra RTVE de documentos desclasificados](https://www.rtve.es/noticias/20260225/todos-documentos-desclasificados-23f-lista-integra/16954095.shtml)**  
   Fuente complementaria de RTVE. Se usará principalmente para localizar enlaces directos a PDFs u originales digitalizados.

3. **[Página oficial de La Moncloa sobre la desclasificación de documentos del 23-F](https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx)**  
   Fuente institucional de contraste. Se usará para construir un inventario oficial y verificar si la cobertura de RTVE es completa.

## Criterio de trabajo

No se parte de la idea de que una fuente sea automáticamente mejor que otra. Cada fuente se evaluará según su utilidad concreta:

- RTVE Buscador: metadatos y texto completo.
- RTVE Lista íntegra: localización y descarga de PDFs.
- La Moncloa: validación oficial de cobertura mediante inventario documental.

A partir de esa comparación se decidirá:

- qué fuente será la base principal del dataset;
- qué documentos se podrán analizar;
- si hay documentos oficiales que falten en RTVE;
- qué textos son suficientemente útiles para NLP;
- qué mini casos son técnicamente viables.

In [8]:
# ============================================================
# 0. Configuración inicial del notebook
# ============================================================

from pathlib import Path
import os
import sys
import json
import re
import warnings
from collections import Counter
from urllib.parse import urlparse, urljoin

import requests
from bs4 import BeautifulSoup

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0.1. Localización robusta de la raíz del proyecto
# ------------------------------------------------------------

def find_project_root(start_path: Path) -> Path:
    """
    Busca la raíz del proyecto subiendo desde start_path.

    La raíz se identifica por:
    - presencia de carpeta .git, o
    - presencia simultánea de README.md y notebooks/

    Parameters
    ----------
    start_path : Path
        Ruta inicial desde la que se empieza a buscar.

    Returns
    -------
    Path
        Ruta raíz del proyecto.

    Raises
    ------
    FileNotFoundError
        Si no se encuentra una raíz válida.
    """
    start_path = start_path.resolve()
    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        has_git = (path / ".git").exists()
        has_readme_and_notebooks = (path / "README.md").exists() and (path / "notebooks").exists()

        if has_git or has_readme_and_notebooks:
            return path

    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Ejecuta el notebook desde dentro del repositorio."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

# ------------------------------------------------------------
# 0.2. Definición de rutas relativas del proyecto
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

# Datos originales derivados de RTVE
RAW_PDFS_RTVE_DIR = RAW_DIR / "pdfs_rtve"
RAW_TEXTS_RTVE_DIR = RAW_DIR / "texts_rtve"

# Outputs del proyecto
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

SRC_DIR = PROJECT_ROOT / "src"

# Crear carpetas si no existen
for folder in [
    DATA_DIR,
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    RAW_PDFS_RTVE_DIR,
    RAW_TEXTS_RTVE_DIR,
    OUTPUTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# Permitir importar módulos propios desde src/
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# ------------------------------------------------------------
# 0.3. URLs de trabajo
# ------------------------------------------------------------

URL_RTVE_BUSCADOR = "https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/"
URL_RTVE_LISTA = "https://www.rtve.es/noticias/20260225/todos-documentos-desclasificados-23f-lista-integra/16954095.shtml"
URL_MONCLOA = "https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx"

SOURCE_URLS = {
    "rtve_buscador": URL_RTVE_BUSCADOR,
    "rtve_lista": URL_RTVE_LISTA,
    "moncloa": URL_MONCLOA,
}

# ------------------------------------------------------------
# 0.4. Cabeceras HTTP comunes
# ------------------------------------------------------------

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

# ------------------------------------------------------------
# 0.5. Configuración visual de pandas
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 120)

# ------------------------------------------------------------
# 0.6. Definición de productos de datos esperados
# ------------------------------------------------------------

# Inventarios documentales
INVENTORY_RTVE_PATH = INTERIM_DIR / "inventory_rtve.csv"
INVENTORY_MONCLOA_PATH = INTERIM_DIR / "inventory_moncloa.csv"

# Textos completos de RTVE
DOCUMENT_TEXTS_RTVE_PATH = INTERIM_DIR / "document_texts_rtve.csv"

# Manifiesto de PDFs RTVE
PDF_MANIFEST_RTVE_PATH = INTERIM_DIR / "pdf_manifest_rtve.csv"

# Comparación y tabla final de documentos validados
SOURCE_COMPARISON_PATH = INTERIM_DIR / "source_comparison.csv"
DOCUMENTS_MASTER_PATH = INTERIM_DIR / "documents_master.csv"
DOCUMENTS_CLEAN_PATH = PROCESSED_DIR / "documents_clean.csv"

# Esquema esperado del inventario documental de RTVE
INVENTORY_RTVE_COLUMNS = [
    "doc_id",
    "source",
    "title",
    "pages",
    "summary",
    "keywords",
    "detail_url",
    "pdf_url",
]

# Esquema esperado del inventario de La Moncloa
INVENTORY_MONCLOA_COLUMNS = [
    "moncloa_id",
    "source",
    "title",
    "pdf_url",
]

# Esquema esperado de la tabla de textos completos
DOCUMENT_TEXT_COLUMNS = [
    "doc_id",
    "text_full",
    "text_length_chars",
    "text_length_words",
    "extraction_source",
]

# Esquema esperado del manifiesto de PDFs RTVE
PDF_MANIFEST_RTVE_COLUMNS = [
    "doc_id",
    "source",
    "title",
    "pdf_url",
    "local_pdf_path",
    "download_ok",
    "file_size_bytes",
]

# Esquema esperado de comparación entre fuentes
SOURCE_COMPARISON_COLUMNS = [
    "doc_id",
    "title",
    "in_rtve",
    "in_moncloa",
    "rtve_pdf_url",
    "moncloa_pdf_url",
    "match_status",
    "notes",
]

# Esquema esperado de la tabla final de documentos validados
DOCUMENTS_MASTER_COLUMNS = [
    "doc_id",
    "title",
    "source_primary",
    "in_rtve",
    "in_moncloa",
    "include_in_corpus",
    "inclusion_reason",
    "pages",
    "summary",
    "keywords",
    "detail_url",
    "pdf_url",
    "text_available",
]

# ------------------------------------------------------------
# 0.7. Comprobación inicial
# ------------------------------------------------------------

print("Configuración inicial completada.")
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Carpeta de datos: {DATA_DIR}")
print(f"Carpeta interim: {INTERIM_DIR}")
print(f"Carpeta processed: {PROCESSED_DIR}")
print(f"Carpeta outputs: {OUTPUTS_DIR}")
print()
print("Productos de datos esperados definidos:")
print(f"- Inventario RTVE: {INVENTORY_RTVE_PATH.relative_to(PROJECT_ROOT)}")
print(f"- Textos RTVE: {DOCUMENT_TEXTS_RTVE_PATH.relative_to(PROJECT_ROOT)}")
print(f"- PDFs RTVE: {RAW_PDFS_RTVE_DIR.relative_to(PROJECT_ROOT)}")
print(f"- Inventario Moncloa: {INVENTORY_MONCLOA_PATH.relative_to(PROJECT_ROOT)}")
print(f"- Comparación fuentes: {SOURCE_COMPARISON_PATH.relative_to(PROJECT_ROOT)}")
print(f"- Tabla final de documentos validados: {DOCUMENTS_MASTER_PATH.relative_to(PROJECT_ROOT)}")

Configuración inicial completada.
Raíz del proyecto: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML
Carpeta de datos: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/data
Carpeta interim: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/data/interim
Carpeta processed: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/data/processed
Carpeta outputs: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/outputs

Productos de datos esperados definidos:
- Inventario RTVE: data/interim/inventory_rtve.csv
- Textos RTVE: data/interim/document_texts_rtve.csv
- PDFs RTVE: data/raw/pdfs_rtve
- Inventario Moncloa: data/interim/inventory_moncloa.csv
- Comparación fuentes: data/interim/source_comparison.csv
- Tabla final de documentos validados: data/interim/documents_master.csv


## 1. Comprobación inicial de fuentes

Antes de construir inventarios, descargar PDFs o extraer textos, se comprueba si las tres fuentes principales son accesibles desde Python.

Esta comprobación es necesaria porque cada producto de datos depende de una fuente distinta:

- el inventario documental y el texto completo dependen principalmente del Buscador RTVE;
- los PDFs de RTVE dependen principalmente de la lista íntegra de RTVE;
- la validación de cobertura depende del inventario oficial de La Moncloa.

El objetivo de esta sección no es extraer todavía todos los documentos, sino verificar:

- si cada página responde correctamente;
- si devuelve contenido HTML;
- si podemos leer su título;
- si alguna fuente requiere un tratamiento técnico especial;
- si tiene sentido continuar con la extracción automática.

Las fuentes evaluadas son:

1. Buscador RTVE 23-F.
2. Lista íntegra RTVE de documentos desclasificados.
3. Página oficial de La Moncloa.

In [9]:
# ============================================================
# 1. Comprobación inicial de acceso a las fuentes
# ============================================================

def fetch_html(url: str, timeout: int = 20) -> dict:
    """
    Descarga el HTML de una URL y devuelve información básica de diagnóstico.

    Parameters
    ----------
    url : str
        URL que se quiere consultar.
    timeout : int
        Tiempo máximo de espera en segundos.

    Returns
    -------
    dict
        Diccionario con estado de la petición, título HTML y longitud del contenido.
    """
    result = {
        "url": url,
        "domain": urlparse(url).netloc,
        "status_code": None,
        "ok": False,
        "content_type": None,
        "html_length": 0,
        "title": None,
        "error": None,
    }

    try:
        response = requests.get(url, headers=REQUEST_HEADERS, timeout=timeout)
        result["status_code"] = response.status_code
        result["ok"] = response.ok
        result["content_type"] = response.headers.get("Content-Type")
        result["html_length"] = len(response.text) if response.text else 0

        soup = BeautifulSoup(response.text, "html.parser")
        title_tag = soup.find("title")
        result["title"] = title_tag.get_text(strip=True) if title_tag else None

    except Exception as exc:
        result["error"] = str(exc)

    return result


source_diagnostics = []

for source_name, source_url in SOURCE_URLS.items():
    info = fetch_html(source_url)
    info["source_name"] = source_name
    source_diagnostics.append(info)

df_source_diagnostics = pd.DataFrame(source_diagnostics)

cols = [
    "source_name",
    "domain",
    "status_code",
    "ok",
    "content_type",
    "html_length",
    "title",
    "error",
    "url",
]

df_source_diagnostics = df_source_diagnostics[cols]

df_source_diagnostics

,source_name,domain,status_code,ok,content_type,html_length,title,error,url
0,rtve_buscador,www.rtve.es,200,True,text/html; charset=ISO-8859-1,42506,Buscador 23F: explora los documentos desclasificados,None,https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/
1,rtve_lista,www.rtve.es,200,True,text/html; charset=utf-8,104862,Documentos desclasificados 23f: lista completa con los archivos,None,https://www.rtve.es/noticias/20260225/todos-documentos-desclasificados-23f-lista-integra/16954095.shtml
2,moncloa,www.lamoncloa.gob.es,200,True,text/html; charset=utf-8,149765,La Moncloa. Documentos desclasificados relativos al 23-F | 25/02/2026 [Consejo de Ministros],None,https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx


# 2. Exploración de la estructura HTML del Buscador RTVE

El Buscador RTVE 23-F es la fuente principal para construir el inventario documental y obtener el texto completo de cada documento.

A partir de esta fuente se intentarán extraer dos productos:

1. `inventory_rtve.csv`  
   Tabla ligera con metadatos: `doc_id`, título, páginas, resumen, palabras clave, URL de detalle y posible URL del PDF.

2. `document_texts_rtve.csv` y archivos `.txt`  
   Tabla y ficheros separados con el texto completo de cada documento, relacionados con el inventario mediante `doc_id`.

La separación entre inventario y texto completo es importante porque el inventario debe ser una tabla manejable de metadatos, mientras que el texto completo puede ser largo y se podría utilizar después para NLP, clustering, grafos, búsqueda semántica y análisis de calidad textual.

En esta sección todavía no se construye el inventario completo. Primero se inspecciona la estructura de la página para comprobar:

1. si existen tablas HTML;
2. si existen filas o bloques repetidos de documentos;
3. si los datos aparecen directamente en el HTML inicial;
4. si hay enlaces internos hacia páginas de detalle;
5. si la web usa JavaScript, JSON o algún endpoint/API interno para cargar los resultados.

Esta inspección es necesaria porque, si los documentos no aparecen directamente en el HTML descargado con `requests`, habrá que localizar la fuente dinámica de datos antes de extraer el inventario completo.

In [10]:
# ============================================================
# 2. Exploración de la estructura HTML del Buscador RTVE
# ============================================================

def get_soup(url: str, timeout: int = 20) -> BeautifulSoup:
    """
    Descarga una página HTML y devuelve un objeto BeautifulSoup.

    Parameters
    ----------
    url : str
        URL de la página.
    timeout : int
        Tiempo máximo de espera.

    Returns
    -------
    BeautifulSoup
        HTML parseado.
    """
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=timeout)
    response.raise_for_status()

    # Si el servidor no declara codificación, se usa la detectada por requests.
    if response.encoding is None:
        response.encoding = response.apparent_encoding

    return BeautifulSoup(response.text, "html.parser")


soup_rtve_buscador = get_soup(URL_RTVE_BUSCADOR)

print("HTML descargado correctamente.")
print(f"Título HTML: {soup_rtve_buscador.title.get_text(strip=True) if soup_rtve_buscador.title else None}")
print(f"Número de caracteres del HTML: {len(str(soup_rtve_buscador)):,}")

HTML descargado correctamente.
Título HTML: Buscador 23F: explora los documentos desclasificados
Número de caracteres del HTML: 29,886


In [11]:
# ------------------------------------------------------------
# 2.1. Comprobación de tablas HTML
# ------------------------------------------------------------

tables = soup_rtve_buscador.find_all("table")

print(f"Número de tablas HTML encontradas: {len(tables)}")

for i, table in enumerate(tables[:5], start=1):
    rows = table.find_all("tr")
    print(f"\nTabla {i}: {len(rows)} filas")

    headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]
    if headers:
        print("Cabeceras:", headers)

    first_rows = []
    for row in rows[:3]:
        cells = [cell.get_text(" ", strip=True) for cell in row.find_all(["td", "th"])]
        first_rows.append(cells)

    print("Primeras filas:")
    for row in first_rows:
        print(row)

Número de tablas HTML encontradas: 0


In [12]:
# ------------------------------------------------------------
# 2.2. Comprobación de si los documentos aparecen en el HTML
# ------------------------------------------------------------

page_text = soup_rtve_buscador.get_text(" ", strip=True)

search_terms = [
    "Vista oral",
    "Consejo Supremo",
    "Resumen",
    "Palabras clave",
    "167 resultados",
    "Texto completo",
]

for term in search_terms:
    found = term.lower() in page_text.lower()
    print(f"'{term}': {found}")

'Vista oral': False
'Consejo Supremo': False
'Resumen': True
'Palabras clave': True
'167 resultados': False
'Texto completo': False


In [13]:
# ------------------------------------------------------------
# 2.3. Inspección de enlaces internos
# ------------------------------------------------------------

links = []

for a in soup_rtve_buscador.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = urljoin(URL_RTVE_BUSCADOR, a["href"])

    if text or "23f" in href.lower():
        links.append({
            "text": text,
            "href": href,
        })

df_links_rtve_buscador = pd.DataFrame(links).drop_duplicates()

print(f"Número de enlaces detectados: {len(df_links_rtve_buscador)}")

df_links_rtve_buscador.head(30)

Número de enlaces detectados: 109


,text,href
0,Saltar al contenido principal,https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/#mainContent
1,Saltar al menú de contenido,https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/#nearNavmenu
2,Saltar al pie de página,https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/#mainFooter
3,Volver a navegación principal,https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/#mainNavmenu
4,Jannik Sinner,https://www.rtve.es/deportes/20260424/mutua-madrid-open-2026-sinner-primera-ronda-entrevista/17035929.shtml
5,Atentado Trump,https://www.rtve.es/noticias/20260427/intento-atentado-claves-reconstruccion-ataque-trump/17042168.shtml
6,Eclipse solar,https://www.rtve.es/noticias/20260428/como-hacer-ensayo-general-del-eclipse-total-del-12-agosto-prueba-clave-para-no...
7,Melania Trump,https://www.rtve.es/noticias/20260427/matrimonio-trump-carga-jimmy-kimmel-parodia-cena-corresponsales/17043692.shtml
8,Apagón 2025,https://www.rtve.es/noticias/20260428/apagon-ano-despues-incidente-inedito-fenomeno-multifactorial-responsabilidad-c...
9,Huelga médicos,https://www.rtve.es/noticias/20260427/medicos-exigen-sanchez-intervencion-directa-semanas-huelga-nacional/17042697.s...


In [14]:
# ------------------------------------------------------------
# 2.4. Inspección de clases CSS frecuentes
# ------------------------------------------------------------

class_counter = Counter()

for tag in soup_rtve_buscador.find_all(True):
    classes = tag.get("class")
    if classes:
        for cls in classes:
            class_counter[cls] += 1

df_classes = (
    pd.DataFrame(class_counter.items(), columns=["class", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

df_classes.head(40)

,class,count
0,ico,65
1,beoff,16
2,blindBox,12
3,tabH1,11
4,wrapper,11
5,brows,9
6,blind,7
7,container,7
8,legend,7
9,datnum,4
